# Kaggle runner: Week 1 (reproduce Multi-GIN + EU on HI-Small)

Before running: **Accelerator = GPU**, **Internet = On**, and add the Kaggle dataset
`ealtman2019/ibm-transactions-for-anti-money-laundering-aml` as an input.

Safe by default: `RUN_FULL = False` runs a 3-epoch smoke test: train, score, write a record
(with a `_smoke` run id, marked `smoke_test` in the record).
Set it to `True` only after the owner has approved the GPU time (AGENTS.md rule 6).
Untested on Kaggle so far: the first run may need small fixes.

In [ ]:
import os, time

REPO_URL = "https://github.com/coachuttam/aml-gnn-transfer.git"
KAGGLE_DATA = "/kaggle/input/ibm-transactions-for-anti-money-laundering-aml"
RUN_FULL = False  # False = smoke test only; True = full reproduction run
WORK = "/kaggle/working"

os.environ["AML_DATA_ROOT"] = f"{WORK}/data"
os.environ["AML_MODEL_ROOT"] = f"{WORK}/models"
# Keep __pycache__ out of the Multi-GNN submodule folder.
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

In [ ]:
!git clone --recurse-submodules {REPO_URL} {WORK}/repo
%cd {WORK}/repo
!git -C external/Multi-GNN rev-parse HEAD

import sys
sys.path.insert(0, f"{WORK}/repo")
from src.config import MULTI_GNN_ADAPTATION_FLAGS, REPRODUCTION_EPOCHS, SEEDS, SMOKE_TEST_EPOCHS
from src.records import make_run_id, smoke_run_id

# One run id names the run folder, its logs and its model folder.
RUN_ID = make_run_id("gin", "source", 1.0, SEEDS[0])
if not RUN_FULL:
    RUN_ID = smoke_run_id(RUN_ID)  # smoke runs never share a real run's id or folders
print("Run id:", RUN_ID)

Multi-GNN needs Python 3.9 and CUDA 11.8 packages, so we build its own `env.yml`
with micromamba (about 5-10 minutes).

In [ ]:
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C {WORK} bin/micromamba
!{WORK}/bin/micromamba create -y -r {WORK}/mamba -n multignn -f external/Multi-GNN/env.yml

If micromamba failed (no `multignn` environment), this cell falls back to Kaggle's own
Python and torch plus `torch_geometric`. Package versions then differ from `env.yml`,
so say so when reporting results.

In [ ]:
ENV_OK = os.path.isdir(f"{WORK}/mamba/envs/multignn")
if ENV_OK:
    RUN = f"{WORK}/bin/micromamba run -r {WORK}/mamba -n multignn"
else:
    import torch
    torch_version = torch.__version__.split("+")[0]
    cuda_tag = "cu" + torch.version.cuda.replace(".", "")
    wheels = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
    get_ipython().system(f"pip install -q torch_geometric datatable munch wandb")
    get_ipython().system(f"pip install -q torch_scatter torch_sparse -f {wheels}")
    RUN = ""  # use the notebook's own python
ENV_LABEL = "env.yml" if ENV_OK else "pip-fallback"
print("Using:", ENV_LABEL)

Format HI-Small and LI-Small with IBM's script and write `data_config.json` (CPU only, slow).

In [ ]:
!{RUN} python -m src.prepare --raw-dir {KAGGLE_DATA} --run-id {RUN_ID}

Multi-GIN + EU on HI-Small. Seed and flags come from `src/config.py`. Model and logs go in folders named by run id.

In [ ]:
EPOCHS = REPRODUCTION_EPOCHS if RUN_FULL else SMOKE_TEST_EPOCHS
FLAGS = " ".join(MULTI_GNN_ADAPTATION_FLAGS)
MAIN = f"{WORK}/repo/external/Multi-GNN/main.py"

# Start from the run folder so Multi-GNN reads OUR data_config.json.
%cd {WORK}/data/multignn_runs/{RUN_ID}
start = time.time()
!{RUN} python {MAIN} --data Small_HI --model gin {FLAGS} --seed {SEEDS[0]} --n_epochs {EPOCHS} --testing --save_model --unique_name
RUNTIME_SECONDS = time.time() - start
print(f"{EPOCHS} epochs took {RUNTIME_SECONDS / 60:.1f} min in total (includes data loading)")

Score the saved checkpoint (threshold chosen on validation, test scored once) and write
the run record. The record lands in `repo/results/runs/`; download it from the
notebook output and commit it. The training run's own log is in the run folder.

In [ ]:
%cd {WORK}/repo
SMOKE = "" if RUN_FULL else "--smoke-test"
!{RUN} python -m src.record_stock_run --run-id {RUN_ID} --model gin --dataset HI-Small --seed {SEEDS[0]} --epochs {EPOCHS} --runtime-seconds {RUNTIME_SECONDS:.0f} --environment {ENV_LABEL} {SMOKE}
!cat results/runs/{RUN_ID}.json

Per-epoch scores. Pick the epoch with the best **validation** F1 and read the test F1
from that same epoch (never the best test epoch).

In [ ]:
!grep -E "Validation F1|Test F1" {WORK}/data/multignn_runs/{RUN_ID}/logs/logs.log